# Data Cleaning & Preprocessing
- **Description:** Preparing the Netflix dataset for analysis by cleaning, transforming and organizing the raw data.
- Task 1
- workflow
1. Import the dataset using Pandas
2. Identify missing and duplicate records
3. Handle null values and inconsistent formats
4. Transform categorical and date-related columns
5. Create a clean dataset for further analysis

## Step 1: Importing dataset using Pandas

In [1]:
import pandas as pd

data = pd.read_csv("Dataset.csv")
print(data.shape)
data.head()

(8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 1.6 MB


## Step 2: Identifying missing and duplicate records

In [3]:
# Empty (NaN) values in each column
print(data.isna().sum())

# Duplicate rows
print("Duplicates:", data.duplicated().sum())

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64
Duplicates: 0


In [4]:
# Some missing values are written as the text "Not Given", so isna() cannot see them
print((data[["director", "country"]] == "Not Given").sum())

director    2588
country      287
dtype: int64


## Step 3: Handling null values and inconsistent formats

In [5]:
# Replace "Not Given" with "Unknown"
data[["director", "country"]] = data[["director", "country"]].replace("Not Given", "Unknown")

# Remove duplicate rows
data = data.drop_duplicates()

# Convert date_added from text to a real date
data["date_added"] = pd.to_datetime(data["date_added"])

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8790 non-null   str           
 1   type          8790 non-null   str           
 2   title         8790 non-null   str           
 3   director      8790 non-null   str           
 4   country       8790 non-null   str           
 5   date_added    8790 non-null   datetime64[us]
 6   release_year  8790 non-null   int64         
 7   rating        8790 non-null   str           
 8   duration      8790 non-null   str           
 9   listed_in     8790 non-null   str           
dtypes: datetime64[us](1), int64(1), str(8)
memory usage: 1.5 MB


## Step 4: Transforming categorical and date-related columns

In [6]:
# Date columns
data["year_added"] = data["date_added"].dt.year
data["month_added"] = data["date_added"].dt.month
data["years_to_netflix"] = (data["year_added"] - data["release_year"]).clip(lower=0)

# Duration: take the number only ("90 min" -> 90, "2 Seasons" -> 2)
data["duration_value"] = data["duration"].str.split().str[0].astype(int)

# Country and genres: count them and keep the first country
data["primary_country"] = data["country"].str.split(",").str[0]
data["country_count"] = data["country"].str.split(",").str.len()
data["genre_count"] = data["listed_in"].str.split(",").str.len()

# Group ratings into audience types
audience = {"TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
            "TV-PG": "Older Kids", "PG": "Older Kids",
            "TV-14": "Teens", "PG-13": "Teens",
            "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults"}
data["audience"] = data["rating"].map(audience).fillna("Not Rated")

data[["type", "duration", "duration_value", "year_added", "primary_country", "audience"]].head()

,type,duration,duration_value,year_added,primary_country,audience
0,Movie,90 min,90,2021,United States,Teens
1,TV Show,1 Season,1,2021,France,Adults
2,TV Show,1 Season,1,2021,United States,Adults
3,Movie,91 min,91,2021,Brazil,Older Kids
4,Movie,125 min,125,2021,United States,Adults


## Step 5: Creating a clean dataset for further analysis

In [7]:
data.to_csv("Cleaned_Netflix_Dataset.csv", index=False)

print("Clean dataset shape:", data.shape)
print("Missing values left:", data.isna().sum().sum())

Clean dataset shape: (8790, 18)
Missing values left: 0


## Summary
- Checked missing values and duplicates, and found hidden missing values ("Not Given").
- Replaced them with "Unknown" and converted `date_added` to a date.
- Created new columns: `year_added`, `month_added`, `years_to_netflix`, `duration_value`, `primary_country`, `country_count`, `genre_count`, `audience`.
- Saved the result as `Cleaned_Netflix_Dataset.csv`.